# NeuroLipin — Multi-Level Structural & Functional Annotation of Neuropilin-2 (NRP2)
[![Website](https://img.shields.io/badge/website-github.io-teal)](https://harshini-suresha.github.io/neurolipin/) [![GitHub](https://img.shields.io/badge/github-repo-black)](https://github.com/Harshini-Suresha/neurolipin) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Harshini-Suresha/neurolipin/blob/main/NeuroLipin.ipynb)

Interactive companion to the static site: https://harshini-suresha.github.io/neurolipin/

- **Live site (GitHub Pages):** https://harshini-suresha.github.io/neurolipin/
- **Repo:** https://github.com/Harshini-Suresha/neurolipin
- **This notebook in Colab:** https://colab.research.google.com/github/Harshini-Suresha/neurolipin/blob/main/NeuroLipin.ipynb
- **UniProt:** [O60462 (NRP2 human)](https://www.uniprot.org/uniprotkb/O60462)

Runs in Colab with zero setup (`numpy` + `matplotlib` + `requests` only).

## 0. Setup

In [ ]:
!pip -q install matplotlib numpy requests
import numpy as np, matplotlib.pyplot as plt, re
from collections import Counter
print('ok')

## 1. Get NRP2 sequence
Paste FASTA/raw, or fetch UniProt O60462 (human NRP2). Falls back to a demo N-terminal fragment if offline.

In [ ]:
import requests
DEMO_SEQ = 'MKLLVLALLCAWPAGAQEDQGAEHKRSTPNFYWMKLLIVAVFGLLLAAIVLFWRRKKEDQNSTQEHKRDEEPSSGGHHYQFLTSTRKMTEEGLPAVNDEKRQ'
def fetch_uniprot(acc='O60462'):
    try:
        r = requests.get(f'https://rest.uniprot.org/uniprotkb/{acc}.fasta', timeout=20)
        r.raise_for_status()
        seq = ''.join(r.text.splitlines()[1:]).strip().upper()
        seq = re.sub('[^ACDEFGHIKLMNPQRSTVWY]','',seq)
        print(f'fetched {acc}: {len(seq)} aa')
        return seq
    except Exception as e:
        print('fetch failed, using demo fragment:', e)
        return DEMO_SEQ
SEQ = fetch_uniprot('O60462')
print(len(SEQ), SEQ[:60])


## 2. ProtParam-style properties (MW, pI, GRAVY, aliphatic index, composition)

In [ ]:
MW = {'A':71.0788,'R':156.1875,'N':114.1038,'D':115.0886,'C':103.1388,'E':129.1155,'Q':128.1307,'G':57.0519,'H':137.1411,'I':113.1594,'L':113.1594,'K':128.1741,'M':131.1926,'F':147.1766,'P':97.1167,'S':87.0782,'T':101.1051,'W':186.2132,'Y':163.1760,'V':99.1326}
KD = {'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'E':-3.5,'Q':-3.5,'G':-0.4,'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,'S':-0.8,'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2}
def theoretical_pi(counts):
    def q(h):
        p = lambda x: 1/(1+10**(h-x)); n = lambda x: 1/(1+10**(x-h))
        return p(9)+counts.get('K',0)*p(10.5)+counts.get('R',0)*p(12.4)+counts.get('H',0)*p(6)-n(3.1)-counts.get('D',0)*n(3.9)-counts.get('E',0)*n(4.1)-counts.get('C',0)*n(8.5)-counts.get('Y',0)*n(10.1)
    a,b=0.0,14.0
    for _ in range(40):
        m=(a+b)/2
        if q(m)>0: a=m
        else: b=m
    return (a+b)/2
L=len(SEQ); c=Counter(SEQ)
mw=sum(MW[a] for a in SEQ)+18.015
gravy=sum(KD[a] for a in SEQ)/L
pi=theoretical_pi(c)
aliph=(c.get('A',0)+2.9*c.get('V',0)+3.9*(c.get('I',0)+c.get('L',0)))/L*100
print(f'length {L} aa | MW {mw/1000:.2f} kDa | pI {pi:.2f} | GRAVY {gravy:.3f} | aliphatic {aliph:.1f}')
aas=sorted(MW, key=lambda a:-c.get(a,0))
plt.figure(figsize=(10,3.5)); plt.bar([a for a in aas],[c.get(a,0)/L*100 for a in aas])
plt.ylabel('%'); plt.title('Amino-acid composition'); plt.tight_layout(); plt.show()


## 3. Hydropathy plot (Kyte-Doolittle, window 19) + TM candidates

In [ ]:
W=19
prof=np.array([np.mean([KD[a] for a in SEQ[i:i+W]]) for i in range(len(SEQ)-W+1)])
plt.figure(figsize=(11,3.5)); plt.plot(np.arange(len(prof))+W//2, prof, lw=1.5)
plt.axhline(1.6, color='red', ls='--', label='TM threshold 1.6'); plt.axhline(0,color='gray',lw=0.8)
tm=np.where(prof>1.6)[0]
for i in tm: plt.axvspan(i, i+1, color='red', alpha=0.12)
plt.xlabel('residue'); plt.ylabel('mean KD'); plt.title(f'Hydropathy (w={W}), {len(tm)} TM-grade windows')
plt.legend(); plt.tight_layout(); plt.show()
print('TM-grade window starts (1-based):', (tm[:20]+1).tolist(), '...' if len(tm)>20 else '')


## 4. Domain architecture (schematic, ~930 aa full-length NRP2)

In [ ]:
domains=[('a1\nCUB1',20,112,'#0d9488'),('a2\nCUB2',120,212,'#14b8a6'),('b1\nFV/VIII',220,312,'#6366f1'),('b2\nFV/VIII',320,412,'#818cf8'),('c\nMAM',420,512,'#f59e0b'),('TM',770,804,'#ec4899'),('C-tail',818,880,'#64748b')]
fig,ax=plt.subplots(figsize=(11,2.6)); ax.set_xlim(0,930); ax.set_ylim(0,3)
ax.plot([0,930],[1,1],color='#ccc')
for name,s,e,col in domains:
    ax.barh(1,e-s,left=s,height=1.2,color=col,edgecolor='white'); ax.text((s+e)/2,1,name,ha='center',va='center',color='white',fontsize=8,weight='bold')
ax.set_xlabel('residue (approx, full-length ~930 aa)'); ax.set_yticks([]); ax.set_title('NRP2 domain map — boundaries approximate, replace with InterProScan/CDD')
plt.tight_layout(); plt.show()


## 5. Interactors (curated list from the site) + network

In [ ]:
import math
interactors=[('VEGFA','Ligand','Binds b1/b2; pro-angiogenic'),('VEGFC','Ligand','Lymphangiogenesis'),('VEGFD','Ligand','Lymphatic signalling'),('SEMA3F','Semaphorin','High-affinity; axon guidance'),('SEMA3C','Semaphorin','Guidance cue'),('SEMA3G','Semaphorin','Vascular guidance'),('PLXNA1','Plexin','Signal-transducing co-receptor'),('PLXNA2','Plexin','Co-receptor'),('PLXNA3','Plexin','Neuronal guidance'),('PLXNA4','Plexin','Neuronal guidance'),('NRP1','Receptor','Paralog; heterodimer'),('KDR','Receptor','Angiogenic complex'),('FLT4','Receptor','Lymphatic complex'),('GIPC1','Adaptor','Binds C-tail PDZ')]
for x in interactors: print(f'{x[0]:8s} {x[1]:10s} {x[2]}')
colors={'Ligand':'#6366f1','Semaphorin':'#f59e0b','Plexin':'#d97706','Receptor':'#ec4899','Adaptor':'#64748b'}
fig,ax=plt.subplots(figsize=(7,7)); ax.set_aspect('equal'); ax.axis('off')
ax.scatter([0],[0],s=2500,color='#0d9488'); ax.text(0,0,'NRP2',ha='center',va='center',color='white',weight='bold')
n=len(interactors)
for i,(name,cls,_) in enumerate(interactors):
    a=2*math.pi*i/n; x,y=3.2*math.cos(a),3.2*math.sin(a)
    ax.plot([0,x],[0,y],color='#ccc'); ax.scatter([x],[y],s=900,color=colors[cls]); ax.text(x,y,name,ha='center',va='center',color='white',fontsize=7,weight='bold')
ax.set_title('NRP2 interactome (star layout)'); plt.show()


## 6. Protease cleavage counts (like PeptideCutter-lite)

In [ ]:
def count(pat): return len(re.findall(pat, SEQ))
rows={'Trypsin K/R (not before P)':len(re.findall(r'[KR](?!P)',SEQ)),'Chymotrypsin F/Y/W':sum(SEQ.count(a) for a in 'FYW'),'Glu-C (E)':SEQ.count('E'),'Asp-N (D)':SEQ.count('D'),'Lys-C (K)':SEQ.count('K'),'CNBr (M)':SEQ.count('M')}
for k,v in rows.items(): print(f'{k:28s} {v}')
plt.figure(figsize=(8,3)); plt.bar(list(rows),list(rows.values())); plt.xticks(rotation=20,ha='right'); plt.ylabel('sites'); plt.title('Cleavage-site counts'); plt.tight_layout(); plt.show()


## 7. What to open next
- Static site: https://harshini-suresha.github.io/neurolipin/ (Live Sequence Lab, force-directed interactome, 3D explorer, generative art)
- UniProt O60462, STRING, InterPro, RCSB PDB, DeepTMHMM — linked in the site footer.